In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- FACT: gold.fact_inventory_movement
-- =====================================================
 
CREATE TABLE IF NOT EXISTS gold.fact_inventory_movement
(
    stock_move_line_id INT,
 
    product_id INT,
    warehouse_id INT,
    lot_id INT,
    picking_id INT,
 
    from_location_id STRING,
    to_location_id STRING,
 
    quantity DOUBLE,
    unit_price DOUBLE,
    move_value DOUBLE,
 
    lot_name STRING,
    picking_name STRING,
    move_state STRING,
    picking_state STRING,
 
    is_in BOOLEAN,
    is_out BOOLEAN,
    is_internal BOOLEAN,
 
    movement_date DATE,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
 
    dwh_loaded_at TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.fact_inventory_movement AS target
 
USING
(
    SELECT
        sml.stock_move_line_id AS stock_move_line_id,
 
        sm.product_id,
        sm.warehouse_id,
        sml.lot_id,
        sp.stock_picking_id AS picking_id,
 
        sm.location_id       AS from_location_id,
        sm.location_dest_id  AS to_location_id,
 
        CAST(sml.quantity_product_uom AS DOUBLE)                              AS quantity,
        CAST(sm.price_unit AS DOUBLE)                                         AS unit_price,
        CAST(sml.quantity_product_uom AS DOUBLE) * CAST(sm.price_unit AS DOUBLE) AS move_value,
 
        sml.lot_name,
        sp.stock_picking_name              AS picking_name,
        sm.state             AS move_state,
        sp.state             AS picking_state,
 
        CAST(sm.is_in AS BOOLEAN)  AS is_in,
        CAST(sm.is_out AS BOOLEAN) AS is_out,
        (NOT CAST(sm.is_in AS BOOLEAN) AND NOT CAST(sm.is_out AS BOOLEAN)) AS is_internal,
 
        CAST(sml.date AS DATE)        AS movement_date,
        sml.created_at,
        sml.updated_at,
 
        current_timestamp() AS dwh_loaded_at
 
    FROM silver.stock_move_line sml
    LEFT JOIN silver.stock_move sm
        ON sm.stock_move_id = sml.move_id
    LEFT JOIN silver.stock_picking sp
        ON sp.stock_picking_id = sml.picking_id
 
    WHERE sml.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.fact_inventory_movement),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
 
ON target.stock_move_line_id = source.stock_move_line_id
 
WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.product_id       = source.product_id,
    target.warehouse_id     = source.warehouse_id,
    target.lot_id           = source.lot_id,
    target.picking_id       = source.picking_id,
    target.from_location_id = source.from_location_id,
    target.to_location_id   = source.to_location_id,
    target.quantity         = source.quantity,
    target.unit_price       = source.unit_price,
    target.move_value       = source.move_value,
    target.lot_name         = source.lot_name,
    target.picking_name     = source.picking_name,
    target.move_state       = source.move_state,
    target.picking_state    = source.picking_state,
    target.is_in            = source.is_in,
    target.is_out           = source.is_out,
    target.is_internal      = source.is_internal,
    target.movement_date    = source.movement_date,
    target.created_at       = source.created_at,
    target.updated_at       = source.updated_at,
    target.dwh_loaded_at    = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;
 


In [0]:
use catalog smart_odoo;

-- =====================================================
-- DIM: gold.dim_inventory_location
-- =====================================================
 
CREATE TABLE IF NOT EXISTS gold.dim_inventory_location
(
    location_id INT,
    warehouse_id INT,
 
    location_name STRING,
    warehouse_name STRING,
    location_path STRING,
    location_type STRING,
 
    is_active BOOLEAN,
 
    created_at DATE,
    updated_at DATE,
 
    dwh_loaded_at TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.dim_inventory_location AS target
 
USING
(
    SELECT
        sl.stock_location_id    AS location_id,
        sw.warehouse_id   AS warehouse_id,
 
        sl.stock_location_name  AS location_name,
        sw.warehouse_name       AS warehouse_name,
        sl.complete_name        AS location_path,
        sl.usage                AS location_type,
 
        CAST(sl.active AS BOOLEAN) AS is_active,
 
        CAST(sl.created_at AS DATE) AS created_at,
        CAST(sl.updated_at AS DATE) AS updated_at,
 
        current_timestamp() AS dwh_loaded_at
 
    FROM silver.stock_location sl
    LEFT JOIN silver.stock_warehouse sw
        ON sw.warehouse_id = sl.warehouse_id
 
    WHERE sl.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.dim_inventory_location),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
 
ON target.location_id = source.location_id
 
WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.warehouse_id   = source.warehouse_id,
    target.location_name  = source.location_name,
    target.warehouse_name = source.warehouse_name,
    target.location_path  = source.location_path,
    target.location_type  = source.location_type,
    target.is_active      = source.is_active,
    target.updated_at     = source.updated_at,
    target.dwh_loaded_at  = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;
 

In [0]:
USE CATALOG smart_odoo;
-- =====================================================
-- DIM: gold.dim_warehouse
-- =====================================================
 
CREATE TABLE IF NOT EXISTS gold.dim_warehouse
(
    warehouse_id INT,
    company_id INT,
    company_name STRING,
 
    warehouse_name STRING,
    warehouse_code STRING,
    reception_steps STRING,
    delivery_steps STRING,
 
    is_active BOOLEAN,
 
    created_at DATE,
    updated_at DATE,
 
    dwh_loaded_at TIMESTAMP
)
USING DELTA;
 
-- MERGE
 
MERGE INTO gold.dim_warehouse AS target
 
USING
(
    SELECT
        w.warehouse_id AS warehouse_id,
        w.company_id,
        w.company_name,
 
        w.warehouse_name      AS warehouse_name,
        w.code      AS warehouse_code,
        w.reception_steps,
        w.delivery_steps,
 
        CAST(w.active AS BOOLEAN) AS is_active,
 
        CAST(w.created_at AS DATE) AS created_at,
        CAST(w.updated_at AS DATE) AS updated_at,
 
        current_timestamp() AS dwh_loaded_at
 
    FROM silver.stock_warehouse w
 
    WHERE w.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.dim_warehouse),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
 
ON target.warehouse_id = source.warehouse_id
 
WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.company_id      = source.company_id,
    target.company_name    = source.company_name,
    target.warehouse_name  = source.warehouse_name,
    target.warehouse_code  = source.warehouse_code,
    target.reception_steps = source.reception_steps,
    target.delivery_steps  = source.delivery_steps,
    target.is_active       = source.is_active,
    target.updated_at      = source.updated_at,
    target.dwh_loaded_at   = current_timestamp()
 
WHEN NOT MATCHED THEN INSERT *;
